**01_snapshot_inventory (refactored)**

Objetivo: definir un *slice* reproducible, materializar/validar cache local y producir artefactos en `runs/`.

Reglas:
- El notebook **lee local** (`DATA_CACHE_DIR`).
- Cloud (R2) solo se toca mediante scripts (por ejemplo `scripts.r2_sync`) si faltan objetos.
- No se importa `settings` en notebooks (evita bloqueos por variables cloud obligatorias).


In [1]:
import os, sys
from pathlib import Path
from datetime import datetime, timezone

print("python:", sys.executable)
print("cwd:", os.getcwd())
print("time:", datetime.now(timezone.utc).isoformat())

print("\n[env]")
print("DATA_CACHE_DIR:", os.getenv("DATA_CACHE_DIR"))
print("RUNS_DIR:", os.getenv("RUNS_DIR"))
print("R2_BUCKET:", os.getenv("R2_BUCKET"))
print("R2_ENDPOINT:", os.getenv("R2_ENDPOINT"))


python: c:\TSIS_Data\v1\backtest_SmallCaps\backtest\Scripts\python.exe
cwd: c:\TSIS_Data\v1\backtest_SmallCaps\notebooks\01_data_integrity
time: 2026-02-12T19:05:50.317425+00:00

[env]
DATA_CACHE_DIR: None
RUNS_DIR: None
R2_BUCKET: None
R2_ENDPOINT: None


**1) Definir slice y rutas**

In [2]:
import os
from pathlib import Path
from datetime import datetime, timezone

# ---- SLICE (edita aqui) ----
NOTEBOOK_ID = "01_snapshot_inventory_refactored"
SYMBOL = "AABA"
YEAR = 2019
MONTH = 1
ERA_OHLCV = "2019_2025"   # para ohlcv_intraday_1m (si aplica)
FORCE_REBUILD_MANIFEST = False
SLICE_ID = f"{SYMBOL}_{YEAR}_{MONTH:02d}"
# ----------------------------

def detect_project_root() -> Path:
    env_root = os.getenv("PROJECT_ROOT")
    if env_root:
        p = Path(env_root).resolve()
        if (p / "notebooks" / "01_data_integrity").exists() and (p / "scripts" / "build_manifest.py").exists():
            return p

    known = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps")
    if known.exists() and (known / "scripts" / "build_manifest.py").exists():
        return known

    cwd = Path.cwd().resolve()
    for cand in [cwd] + list(cwd.parents):
        if (cand / "notebooks" / "01_data_integrity").exists() and (cand / "scripts" / "build_manifest.py").exists():
            return cand
    return cwd

PROJECT_ROOT = detect_project_root()
os.chdir(PROJECT_ROOT)
os.environ["PYTHONPATH"] = str(PROJECT_ROOT)

PREFIX_OHLCV = f"ohlcv_intraday_1m/{ERA_OHLCV}/{SYMBOL}/year={YEAR}/month={MONTH:02d}/"
PREFIX_QUOTES = f"quotes_p95/{SYMBOL}/year={YEAR}/month={MONTH:02d}/"
TRADES_ERA = "2019_2025" if YEAR >= 2019 else "2004_2018"
PREFIX_TRADES = f"trades_ticks_{TRADES_ERA}/{SYMBOL}/year={YEAR}/month={MONTH:02d}/"

MANIFEST_PATH = PROJECT_ROOT / f"data/manifests/r2_slice_{SYMBOL}_{YEAR}_{MONTH:02d}.json"

DATA_CACHE_DIR = Path(os.environ.get("DATA_CACHE_DIR", r"C:\TSIS_Data\data")).resolve()
RUNS_DIR = Path(os.environ.get("RUNS_DIR", PROJECT_ROOT / "runs")).resolve()
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

print("SLICE_ID:", SLICE_ID)
print("PREFIX_OHLCV:", PREFIX_OHLCV)
print("PREFIX_QUOTES:", PREFIX_QUOTES)
print("PREFIX_TRADES:", PREFIX_TRADES)
print("FORCE_REBUILD_MANIFEST:", FORCE_REBUILD_MANIFEST)
print("MANIFEST_PATH:", MANIFEST_PATH)
print("DATA_CACHE_DIR:", DATA_CACHE_DIR)
print("RUNS_DIR:", RUNS_DIR)
print("RUN_ID:", RUN_ID)


SLICE_ID: AABA_2019_01
PREFIX_OHLCV: ohlcv_intraday_1m/2019_2025/AABA/year=2019/month=01/
PREFIX_QUOTES: quotes_p95/AABA/year=2019/month=01/
PREFIX_TRADES: trades_ticks_2019_2025/AABA/year=2019/month=01/
FORCE_REBUILD_MANIFEST: False
MANIFEST_PATH: C:\TSIS_Data\v1\backtest_SmallCaps\data\manifests\r2_slice_AABA_2019_01.json
DATA_CACHE_DIR: C:\TSIS_Data\data
RUNS_DIR: C:\TSIS_Data\v1\backtest_SmallCaps\runs
RUN_ID: 20260212T190550Z


**2) Build manifest (slice contract)**

In [3]:
import os, subprocess, sys
from pathlib import Path

# Usa PROJECT_ROOT definido en la celda 1
SCRIPT_PATH = PROJECT_ROOT / "scripts" / "build_manifest.py"

MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT)

BUILD_MANIFEST_CMD = [
    sys.executable, str(SCRIPT_PATH),
    "--prefix", PREFIX_OHLCV,
    "--prefix", PREFIX_QUOTES,
    "--prefix", PREFIX_TRADES,
    "--out", str(MANIFEST_PATH),
]

if MANIFEST_PATH.exists() and not FORCE_REBUILD_MANIFEST:
    print("Manifest already exists; skipping build_manifest step:", MANIFEST_PATH)
else:
    if MANIFEST_PATH.exists() and FORCE_REBUILD_MANIFEST:
        print("FORCE_REBUILD_MANIFEST=true -> rebuilding:", MANIFEST_PATH)
    print("BUILD_MANIFEST_CMD:", " ".join(BUILD_MANIFEST_CMD))
    subprocess.check_call(BUILD_MANIFEST_CMD, cwd=str(PROJECT_ROOT), env=env)
    print("OK ->", MANIFEST_PATH)


Manifest already exists; skipping build_manifest step: C:\TSIS_Data\v1\backtest_SmallCaps\data\manifests\r2_slice_AABA_2019_01.json


**3) Snapshot inventory (manifest ↔ cache local)**

In [4]:
import os, subprocess, sys
from pathlib import Path

# Usa PROJECT_ROOT definido en la celda 1
SCRIPT = PROJECT_ROOT / "src" / "data" / "snapshot_inventory.py"

env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT)

def run_snapshot():
    cmd = [
        sys.executable, "-u", str(SCRIPT),
        "--slice-id", SLICE_ID,
        "--manifest", str(MANIFEST_PATH),
        "--cache-dir", str(DATA_CACHE_DIR),
        "--runs-dir", str(RUNS_DIR),
        "--run-id", RUN_ID,
    ]
    print("RUNNING:", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(PROJECT_ROOT), env=env)

run_snapshot()


RUNNING: c:\TSIS_Data\v1\backtest_SmallCaps\backtest\Scripts\python.exe -u C:\TSIS_Data\v1\backtest_SmallCaps\src\data\snapshot_inventory.py --slice-id AABA_2019_01 --manifest C:\TSIS_Data\v1\backtest_SmallCaps\data\manifests\r2_slice_AABA_2019_01.json --cache-dir C:\TSIS_Data\data --runs-dir C:\TSIS_Data\v1\backtest_SmallCaps\runs --run-id 20260212T190550Z


**4) Leer artefactos de snapshot_inventory y fail-fast**

In [5]:
import json
import polars as pl
from pathlib import Path

OUT_INV = RUNS_DIR / "data_quality" / "snapshot_inventory" / SLICE_ID / RUN_ID
print("OUT_INV:", OUT_INV)

summary_path = OUT_INV / "summary.json"
inv_path = OUT_INV / "inventory.parquet"

summary = json.loads(summary_path.read_text(encoding="utf-8"))
inv = pl.read_parquet(inv_path)

print("overall_status:", summary.get("overall_status"))
print("coverage:", summary.get("coverage"))
print("n_missing:", summary.get("n_missing"))
print("n_size_mismatch:", summary.get("n_size_mismatch"))

display(inv.head(10))

if summary.get("overall_status") == "FAIL":
    display(inv.filter(pl.col("status") != "OK").head(50))
    print(f"[WARN] Snapshot inventory FAIL. Revisa: {OUT_INV}")

# Non-interactive mode for nbconvert execution
AUTO_SYNC = str(os.getenv("AUTO_SYNC_R2", "false")).strip().lower() in ("1", "true", "yes", "y")

if summary.get("n_missing", 0) > 0:
    if AUTO_SYNC:
        sync_cmd = [
            sys.executable, str(PROJECT_ROOT / "scripts" / "r2_sync.py"),
            "--manifest", str(MANIFEST_PATH),
        ]
        print("SYNC:", " ".join(sync_cmd))
        subprocess.check_call(sync_cmd, cwd=str(PROJECT_ROOT), env=env)
        run_snapshot()
    else:
        print("[INFO] AUTO_SYNC_R2=false -> skipping interactive r2_sync during automated execution")


OUT_INV: C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\snapshot_inventory\AABA_2019_01\20260212T190550Z
overall_status: PASS
coverage: 1.0
n_missing: 0
n_size_mismatch: 0


key,local_path,exists,local_size,local_mtime_utc,expected_size,expected_etag,expected_last_modified,size_ok,status,dataset,symbol,year,month,day,era
str,str,bool,i64,str,i64,str,str,bool,str,str,str,i64,i64,i64,str
"""ohlcv_intraday_1m/2019_2025/AA…","""C:\TSIS_Data\data\ohlcv_intrad…",true,188753,"""2025-11-25T09:31:38+00:00""",188753,"""86d58162950b13f5d8c9496223b6b4…","""2026-01-20T09:11:38.643000+00:…",true,"""OK""","""ohlcv_intraday_1m""","""AABA""",2019,1,null,"""2019_2025"""
"""quotes_p95/AABA/year=2019/mont…","""C:\TSIS_Data\data\quotes_p95\A…",true,2213211,"""2025-11-26T03:53:49+00:00""",2213211,"""328b36f1cabafac0223d11e8130306…","""2026-01-19T18:36:12.996000+00:…",true,"""OK""","""quotes_p95""","""AABA""",2019,1,2,null
"""quotes_p95/AABA/year=2019/mont…","""C:\TSIS_Data\data\quotes_p95\A…",true,592308,"""2025-11-26T03:53:49+00:00""",592308,"""32ed83b5d389ea330f3bff3ea27498…","""2026-01-19T18:36:19.914000+00:…",true,"""OK""","""quotes_p95""","""AABA""",2019,1,3,null
"""quotes_p95/AABA/year=2019/mont…","""C:\TSIS_Data\data\quotes_p95\A…",true,571009,"""2025-11-26T03:53:50+00:00""",571009,"""ee675dc98f72d61f39a69c1c16a223…","""2026-01-19T18:36:25.037000+00:…",true,"""OK""","""quotes_p95""","""AABA""",2019,1,4,null
"""quotes_p95/AABA/year=2019/mont…","""C:\TSIS_Data\data\quotes_p95\A…",true,2238490,"""2025-11-26T03:53:51+00:00""",2238490,"""0820b3c944f199600386f26ff41014…","""2026-01-19T18:36:35.589000+00:…",true,"""OK""","""quotes_p95""","""AABA""",2019,1,7,null
"""quotes_p95/AABA/year=2019/mont…","""C:\TSIS_Data\data\quotes_p95\A…",true,2031204,"""2025-11-26T03:53:53+00:00""",2031204,"""14e27b55c73ff39bf34958674a0dd1…","""2026-01-19T18:36:41.503000+00:…",true,"""OK""","""quotes_p95""","""AABA""",2019,1,8,null
"""quotes_p95/AABA/year=2019/mont…","""C:\TSIS_Data\data\quotes_p95\A…",true,3066206,"""2025-11-26T03:53:55+00:00""",3066206,"""531f79b01b64282b68a72569a7906b…","""2026-01-19T18:36:48.756000+00:…",true,"""OK""","""quotes_p95""","""AABA""",2019,1,9,null
"""quotes_p95/AABA/year=2019/mont…","""C:\TSIS_Data\data\quotes_p95\A…",true,716123,"""2025-11-26T03:53:56+00:00""",716123,"""cea121253bb3801a18f9067adcacc7…","""2026-01-19T18:36:55.992000+00:…",true,"""OK""","""quotes_p95""","""AABA""",2019,1,10,null
"""quotes_p95/AABA/year=2019/mont…","""C:\TSIS_Data\data\quotes_p95\A…",true,1593334,"""2025-11-26T03:53:57+00:00""",1593334,"""69efa111b3da0c416d5d3d325b10a5…","""2026-01-19T18:37:02.462000+00:…",true,"""OK""","""quotes_p95""","""AABA""",2019,1,11,null


**5) Gate v1 de contenido (validate_data)**

In [6]:
import subprocess, sys
from pathlib import Path

OUT_VALIDATE = RUNS_DIR / "data_quality" / "data_integrity_gate_v1" / SLICE_ID / RUN_ID
OUT_VALIDATE.mkdir(parents=True, exist_ok=True)

VALIDATE_CMD = [
    sys.executable, "-m", "scripts.validate_data",
    "--manifest", str(MANIFEST_PATH),
    "--out_dir", str(OUT_VALIDATE),
]

print("VALIDATE_CMD:\n", " ".join(VALIDATE_CMD))
subprocess.check_call(VALIDATE_CMD)
print("OK ->", OUT_VALIDATE)


VALIDATE_CMD:
 c:\TSIS_Data\v1\backtest_SmallCaps\backtest\Scripts\python.exe -m scripts.validate_data --manifest C:\TSIS_Data\v1\backtest_SmallCaps\data\manifests\r2_slice_AABA_2019_01.json --out_dir C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\data_integrity_gate_v1\AABA_2019_01\20260212T190550Z
OK -> C:\TSIS_Data\v1\backtest_SmallCaps\runs\data_quality\data_integrity_gate_v1\AABA_2019_01\20260212T190550Z


**6) Ver resultados del Gate v1**

In [7]:
import json
from collections import defaultdict

qm_path = OUT_VALIDATE / "quality_metrics.json"
am_path = OUT_VALIDATE / "anomalies.jsonl"

qm = json.loads(qm_path.read_text(encoding="utf-8"))
print("OVERALL:", qm.get("overall"))
print("checked_files:", qm.get("checked_files"))
print("FAIL:", qm.get("fail"), "WARN:", qm.get("warn"), "INFO:", qm.get("info"))

lines = am_path.read_text(encoding="utf-8").splitlines() if am_path.exists() else []
print("anomalies entries:", len(lines))

by_message = defaultdict(int)
rows_by_message = defaultdict(int)

parsed = []
for line in lines:
    a = json.loads(line)
    msg = a.get("message")
    rows = (a.get("details") or {}).get("rows", 0)
    by_message[msg] += 1
    rows_by_message[msg] += rows
    parsed.append(a)

if parsed:
    print("\nSummary by anomaly type:")
    for msg, n in by_message.items():
        print(f"- {msg}: files={n}, rows={rows_by_message[msg]}")
    print("\nExamples:")
    for a in parsed[:5]:
        print(f"- {a.get('message')} | {a.get('dataset')} | {a.get('severity')} | rows={(a.get('details') or {}).get('rows')} | key={a.get('key')}")
else:
    print("No anomalies file or empty.")


OVERALL: WARN
checked_files: 66
FAIL: 0 WARN: 15 INFO: 0
anomalies entries: 15

Summary by anomaly type:
- Crossed market (bid > ask) rows: files=15, rows=229

Examples:
- Crossed market (bid > ask) rows | quotes_p95 | WARN | rows=6 | key=quotes_p95/AABA/year=2019/month=01/day=02/quotes.parquet
- Crossed market (bid > ask) rows | quotes_p95 | WARN | rows=11 | key=quotes_p95/AABA/year=2019/month=01/day=08/quotes.parquet
- Crossed market (bid > ask) rows | quotes_p95 | WARN | rows=3 | key=quotes_p95/AABA/year=2019/month=01/day=09/quotes.parquet
- Crossed market (bid > ask) rows | quotes_p95 | WARN | rows=1 | key=quotes_p95/AABA/year=2019/month=01/day=11/quotes.parquet
- Crossed market (bid > ask) rows | quotes_p95 | WARN | rows=9 | key=quotes_p95/AABA/year=2019/month=01/day=14/quotes.parquet
